# 03 — Local SLM Battery Control · Seasonal-Panel Evaluation · Google Colab

**Phase 2** · MSc Thesis — Supervisor: Dr. Panagiotis Kasnesis · Student: Antonios Bastoulis

---

Evaluates a **local open-weight SLM** (Gemma / Qwen / Llama / Phi) as a zero-shot battery controller on the Colab GPU, against the trained **SAC expert** and rule-based baselines.

**There is no single-week run.** Every policy is scored on a stratified **4-window seasonal panel** (one week per season, 168 steps each) — a single window is a biased estimator and its cost KPI degenerates in solar-rich weather (§ 14). One local SLM per session (GPU VRAM); swap `MODEL_ID` and re-run to compare models. The SLM panel is **resumable** — re-run the § 9 cell after a Colab disconnect and it continues from the last finished window.

## § 0 — Runtime & Config

> **Change experiment parameters here only** — model, the 4-window panel, the SAC Drive path.

In [ ]:
import os, sys, subprocess, time, warnings, json, random
import numpy as np

# ── Runtime check ─────────────────────────────────────────────────────────
try:
    import torch
    if torch.cuda.is_available():
        _g = torch.cuda.get_device_properties(0)
        print(f"GPU: {_g.name}  ({_g.total_memory / 1e9:.1f} GB VRAM)")
    else:
        print("No GPU — Runtime -> Change runtime type -> T4 GPU")
except ImportError:
    print("torch not yet installed — will be in § 1")

# ── Model selection ────────────────────────────────────────────────────────
# Unsloth's Gemma-4 mirror: identical weights to google/gemma-4-E4B-it but not
# gated. Qwen/Qwen3-4B-Instruct-2507 is also a good choice.
MODEL_ID:       str  = "unsloth/gemma-4-E4B-it"
LOAD_IN_4BIT:   bool = True
MAX_NEW_TOKENS: int  = 400

HF_TOKEN:    str = os.environ.get("HF_TOKEN")
GITHUB_REPO: str = "https://github.com/antonisbast/eclipse-thesis"

# ── Single-agent buildings (Phase 3 design — see CLAUDE.md) ───────────────
TRAINING_BUILDINGS: list = [0, 1, 2]
BUILDINGS_RUN:      list = TRAINING_BUILDINGS
N_BLDGS:            int  = len(BUILDINGS_RUN)

# ── Seasonal evaluation panel ─────────────────────────────────────────────
# 4 windows x 7 whole days, whole-day-aligned starts spread across the year.
PANEL: list = [
    {"name": "W1", "start": 1440},
    {"name": "W2", "start": 3624},
    {"name": "W3", "start": 5808},
    {"name": "W4", "start": 7992},
]
PANEL_LEN: int = 168   # 7 whole days

# ── SAC expert — loaded from Drive ────────────────────────────────────────
# Place the trained SAC pickle (sac_*.pkl) in SAC_DRIVE_DIR. It is the learned
# upper bound the SLM is measured against and powers the § 13 calibration.
USE_SAC:       bool = True
SAC_DRIVE_DIR: str  = "/content/drive/MyDrive/eclipse-thesis/agents"

# ── Local output ──────────────────────────────────────────────────────────
LOCAL_OUTDIR: str = "/content/eclipse-thesis/notebooks/artifacts"
os.makedirs(LOCAL_OUTDIR, exist_ok=True)

print(f"\nModel      : {MODEL_ID}  (4-bit={LOAD_IN_4BIT}, max_new_tokens={MAX_NEW_TOKENS})")
print(f"Panel      : {len(PANEL)} windows x {PANEL_LEN} steps  buildings {BUILDINGS_RUN}")
for _w in PANEL:
    print(f"  {_w['name']}: t{_w['start']}..{_w['start'] + PANEL_LEN - 1}")
print(f"SAC expert : {'from ' + SAC_DRIVE_DIR if USE_SAC else 'disabled'}")

## § 1 — Install Dependencies

CityLearn first (`--no-deps`), then the ML stack. `transformers` pinned to 5.5.0 — the version nb 05 trains against.

In [ ]:
# CityLearn 2.6 is a pre-release on PyPI — pin the version explicitly so we
# get the same evaluate_v2() API as the rest of the pipeline (nb 04 / 05 / 06).
# --no-deps because CityLearn pulls heavy/legacy deps we don't need at eval
# time (e.g. some EnergyPlus extras). Runtime deps installed explicitly above.
CITYLEARN_VERSION = "2.6.0b2"

!pip install -q numpy gymnasium doe-xstock nrel-pysam
!pip install -q --pre "CityLearn=={CITYLEARN_VERSION}" --no-deps

import citylearn
assert citylearn.__version__.startswith("2.6"), (
    f"Expected CityLearn 2.6.x, got {citylearn.__version__}. "
    f"src.eval depends on evaluate_v2() which only exists in 2.6+."
)
print(f"✓ CityLearn {citylearn.__version__}")

In [ ]:
# Step 3 — ML stack. transformers is PINNED to 5.5.0 — the version the official
# Unsloth Gemma-4 tutorial requires for Gemma 4 support (nb 05/06 pin the same).
# Installing from GitHub HEAD instead pulls a 5.8.x dev build that drifts from
# the rest of the pipeline.
!pip install -q "transformers==5.5.0" "tokenizers>=0.22.0,<=0.23.0" \
                accelerate bitsandbytes sentencepiece
# timm — Gemma 4 E4B carries vision/audio towers; transformers needs it on load
# even though this notebook only uses the text path.
!pip install -q --no-deps --upgrade timm

import transformers
print(f"✓ ML dependencies installed  |  transformers=={transformers.__version__}")

## § 2 — Clone Repository

Pulls `src/` from GitHub. Run the fresh-clone cell only to discard the local repo and re-pull.

In [ ]:
# ── Fresh clone utility ───────────────────────────────────────────────────
# Run this cell ONLY if you want to discard the local repo and re-clone from
# GitHub (e.g. after pushing new commits). Skip it on a normal Run All.
!rm -rf /content/eclipse-thesis
print("Repo directory removed — run the clone-repo cell (§ 2) next.")

In [ ]:
REPO_DIR = "/content/eclipse-thesis"

if not os.path.exists(REPO_DIR):
    result = subprocess.run(
        ["git", "clone", GITHUB_REPO, REPO_DIR],
        capture_output=True, text=True,
    )
    print(result.stdout or result.stderr)
else:
    result = subprocess.run(
        ["git", "-C", REPO_DIR, "pull"],
        capture_output=True, text=True,
    )
    print("Repo already present —", result.stdout.strip() or "up to date")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"sys.path ← {REPO_DIR}")

# HuggingFace login (only for gated models like Llama)
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("✓ HuggingFace login OK")

In [ ]:
# ── Verify paths (debug) ──────────────────────────────────────────────────
import os, sys
print("Contents of /content/eclipse-thesis:")
try:
    print(os.listdir("/content/eclipse-thesis"))
except FileNotFoundError:
    print("  ✗ Directory not found — did the clone succeed?")

print(f"\nsys.path[0]: {sys.path[0]}")
print(f"src/ present: {os.path.isdir('/content/eclipse-thesis/src')}")

## § 3 — Imports, Environment Factory & SAC Expert

Everything domain-specific from `src/`. `make_colab_env` uses CityLearn's auto-download schema (no local dataset needed). The SAC expert is loaded from Drive — it is the learned upper bound and the calibration reference.

In [ ]:
import logging
import pickle
import glob
import pandas as pd
import matplotlib.pyplot as plt
import citylearn
from citylearn.citylearn import CityLearnEnv

# ── Single source of truth: everything domain-specific from src/ ──────────
from src.env       import (
    SEED, TRAINING_BUILDINGS, snapshot_state,
    MERLINReward, OBSERVATIONS, ACTIVE_ACTIONS,
)
from src.agent     import (
    PRICE_PEAK_THRESHOLD,
    render_state, make_minimal_prompt, make_policy_llm,
    policy_noop, policy_random, policy_rbc,
)
from src.providers import LocalHFProvider
from src.eval      import evaluate
from src.rollout   import (
    run_policy as _run_policy,
    run_sac,
    summarize_district as _summarize,
)

warnings.filterwarnings("ignore")
np.random.seed(SEED)
random.seed(SEED)

BUILDINGS_RUN = TRAINING_BUILDINGS
N_BLDGS       = len(BUILDINGS_RUN)


def make_colab_env(start: int = 0, end: int = 8758, buildings=None) -> CityLearnEnv:
    """Colab variant of src.env.make_env() — CityLearn auto-download schema."""
    env = CityLearnEnv(
        schema="citylearn_challenge_2022_phase_all",
        buildings=buildings if buildings is not None else BUILDINGS_RUN,
        central_agent=False,
        active_actions=ACTIVE_ACTIONS,
        active_observations=OBSERVATIONS,
        random_seed=SEED,
        simulation_start_time_step=start,
        simulation_end_time_step=end,
    )
    env.reward_function = MERLINReward(env.get_metadata())
    return env


def _panel_factory(start, end, obs_set):
    """Env factory for the panel — pinned to BUILDINGS_RUN."""
    return make_colab_env(start=start, end=end)


# ── Load the SAC expert from Drive ────────────────────────────────────────
sac_agent = None
if USE_SAC:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        _pk = sorted(glob.glob(f"{SAC_DRIVE_DIR}/sac_*.pkl"))
        assert _pk, f"No sac_*.pkl in {SAC_DRIVE_DIR}"
        with open(_pk[-1], "rb") as _f:
            sac_agent = pickle.load(_f)
        print(f"SAC expert : {os.path.basename(_pk[-1])}  "
              f"({len(sac_agent.action_names)} policies, device={sac_agent.device})")
    except Exception as _exc:
        print(f"[SAC] not loaded ({_exc}) — panel runs without the expert.")
        sac_agent = None

print(f"CityLearn {citylearn.__version__}  |  make_colab_env ready  |  BUILDINGS_RUN={BUILDINGS_RUN}")
print("First env build downloads the dataset (~50 MB, ~30 s).")

## § 4 — Prompt Preview

`make_minimal_prompt(3)` — the canonical chain-of-thought prompt the local SLM receives.

In [ ]:
print(make_minimal_prompt(N_BLDGS))

## § 5 — Load Model + Warmup

First run downloads model weights from HuggingFace. The warmup cell measures decode throughput on a realistic prompt.

In [ ]:
_t0 = time.time()
slm = LocalHFProvider(
    model_id=MODEL_ID,
    load_in_4bit=LOAD_IN_4BIT,
    max_new_tokens=MAX_NEW_TOKENS,
)
print(f"\nModel loaded in {time.time() - _t0:.1f}s")

In [ ]:
# ── Realistic warmup + speed benchmark ────────────────────────────────────

_SAMPLE_STATE = (
    "Month 12, Wed 10:00  |  price=0.210 (LOW)  |  carbon=0.160 (MID)\n"
    "Buildings:\n"
    "  B0: SoC= 42.5%  load=1.15 kWh  last_net=+0.82 kWh  solar=LOW\n"
    "  B1: SoC= 61.0%  load=0.75 kWh  last_net=+0.51 kWh  solar=LOW\n"
    "  B2: SoC= 28.3%  load=1.48 kWh  last_net=+1.24 kWh  solar=LOW"
)
_WARMUP_PROMPT = make_minimal_prompt(3)

print("Warmup inference (using make_minimal_prompt + realistic state) …")
_wt0  = time.time()
_resp = slm.complete(_WARMUP_PROMPT, f"STATE:\n{_SAMPLE_STATE}", max_tokens=MAX_NEW_TOKENS)
_wt1  = time.time()

_resp_toks = len(slm.tokenizer.encode(_resp, add_special_tokens=False))
_tok_per_s = _resp_toks / (_wt1 - _wt0) if (_wt1 - _wt0) > 0 else 0

print(f"Tokens generated : {_resp_toks}")
print(f"Latency          : {_wt1 - _wt0:.2f}s")
print(f"Throughput       : {_tok_per_s:.0f} tokens/s")
print(f"\nResponse preview:\n{_resp[:400]}{'...' if len(_resp) > 400 else ''}")

## § 6 — Panel Windows

The 4 evaluation windows, one per season — calendar month and the building state at each window start. The **regime** (net-import vs net-export) is filled in from the no-control rollout in § 10; it decides whether the cost KPI is valid there.

In [ ]:
window_month: dict = {}
_snap = None
for _w in PANEL:
    _e = make_colab_env(start=_w["start"], end=_w["start"] + PANEL_LEN - 1)
    _e.reset()
    _s = snapshot_state(_e)
    window_month[_w["name"]] = _s[0]["month"]
    if _snap is None:
        _snap = _s
    print(f"{_w['name']}  t{_w['start']}..{_w['start']+PANEL_LEN-1}  month={_s[0]['month']}")
    for i, d in enumerate(_s):
        print(f"   B{i}: SoC={d['electrical_storage_soc']*100:4.1f}%  "
              f"load={d['non_shiftable_load']:.2f}  price={d['electricity_pricing']:.3f}  "
              f"solar={d['solar_generation']:.2f}")

print("\nAgent state — W1:")
print("=" * 60)
print(render_state(_snap))

## § 7 — Panel Rollout Engine

`panel_rollout()` runs one policy across all 4 windows and stores per-window results in `panel_runs`. It is **resumable**: windows already in `panel_runs[key]` are skipped and the store is saved after every window — a Colab disconnect mid-panel only costs the unfinished windows, so just re-run the cell.

In [ ]:
panel_runs: dict = {}

def panel_rollout(key: str, label: str, kind: str, obj) -> dict:
    """Roll one policy across every PANEL window; store in panel_runs[key].

    Resumable — already-finished windows are skipped.
    kind: 'policy' (policy_fn via run_policy) | 'sac' (SAC agent via run_sac).
    """
    windows = dict(panel_runs.get(key, {}).get("windows", {}))
    for w in PANEL:
        if w["name"] in windows:
            print(f"  {w['name']}: already done — skip")
            continue
        tag = f"{w['name']}_{key}"
        if kind == "sac":
            df, env = run_sac(tag, obj, start=w["start"], length=PANEL_LEN,
                              env_factory=_panel_factory)
            raw = []
        else:
            df, env, raw = _run_policy(tag, obj, start=w["start"], length=PANEL_LEN,
                                       obs_set="llm", env_factory=_panel_factory)
        windows[w["name"]] = {"df": df, "env": env, "raw_log": raw}
        panel_runs[key] = {"label": label, "kind": kind, "windows": windows}  # save each window
    panel_runs[key] = {"label": label, "kind": kind, "windows": windows}
    return panel_runs[key]

print("panel_rollout() ready (resumable). panel_runs initialised.")

## § 8 — Baselines on the Panel

No-op / random / RBC (instant) and the **SAC expert** (a forward pass per step). Run before the SLM cell.

In [ ]:
panel_rollout("noop",   "No Control", "policy", policy_noop)
panel_rollout("random", "Random",     "policy", policy_random)
panel_rollout("rbc",    "RBC",        "policy", policy_rbc)
if sac_agent is not None:
    panel_rollout("sac", "SAC", "sac", sac_agent)
else:
    print("[SAC] skipped — sac_agent not loaded (see § 3).")
print(f"\nBaselines done — panel_runs: {list(panel_runs)}")

## § 9 — Local SLM on the Panel

The local SLM rolled out across all 4 windows. **Resumable** — if Colab disconnects, just re-run this cell and it continues from the last finished window. Budget ~50–60 min per window on an L4 GPU (~4 h for the full panel).

In [ ]:
_SLM_SYSTEM = make_minimal_prompt(N_BLDGS)
_policy     = make_policy_llm(slm, n_buildings=N_BLDGS, agent_label="",
                              system=_SLM_SYSTEM, max_tokens=250)
slm_key = MODEL_ID.split("/")[-1]

print(f"=== {slm.label} on the {len(PANEL)}-window panel "
      f"({len(PANEL) * PANEL_LEN} steps) — resumable ===")
_t0 = time.time()
panel_rollout(slm_key, slm.label, "policy", _policy)
print(f"\nDone in {(time.time() - _t0) / 60:.1f} min — panel_runs['{slm_key}']")

## § 10 — KPI Results

All six Challenge KPIs per window, then aggregated over the **valid-C windows** only. `C` (cost) is a ratio against the no-control baseline; in a net-export window that baseline collapses toward zero and `C` becomes meaningless — those windows are flagged `C valid = NO`. `G`, `R`, `1L` stay well-conditioned everywhere. See § 14.

In [ ]:
KPI_COLS = ["C  — cost", "G  — carbon", "R  — ramping", "1L — load factor",
            "Phase I (C+G)/2", "Combined (C+G+D)/3"]

assert "noop" in panel_runs, "Run the § 8 baselines first."

panel_evals = {}
for _key, _run in panel_runs.items():
    for _w in PANEL:
        panel_evals[(_key, _w["name"])] = evaluate(
            _run["windows"][_w["name"]]["env"], _run["label"])

window_net = {w["name"]: _summarize(panel_runs["noop"]["windows"][w["name"]]["df"],
                                    "noop", n_buildings=N_BLDGS)["total_net_kWh"]
              for w in PANEL}
WINDOW_VALID = {w: net > 0 for w, net in window_net.items()}

print("Window regimes (cost-KPI validity):")
for w in PANEL:
    wn = w["name"]
    print(f"  {wn}  month={window_month[wn]:2d}  no-control net={window_net[wn]:+9.1f} kWh"
          f"  ->  {'import  — C valid' if WINDOW_VALID[wn] else 'EXPORT  — C degenerate'}")

In [ ]:
# Per-window Challenge KPIs — every policy x every window
_recs = []
for _key, _run in panel_runs.items():
    for _w in PANEL:
        wn = _w["name"]
        ev = panel_evals[(_key, wn)]
        _recs.append({
            "window": wn, "month": window_month[wn],
            "C valid": "yes" if WINDOW_VALID[wn] else "NO",
            "policy": _run["label"],
            **{k: round(float(ev.challenge[k]), 3) for k in KPI_COLS},
        })
kpi_panel = pd.DataFrame(_recs).set_index(["window", "policy"]).sort_index()
print("Per-window Challenge KPIs — ratio vs no-control, lower is better (1.0 = no-control)")
display(kpi_panel)

In [ ]:
# Aggregate over valid-C windows + SLM performance as a fraction of SAC
_valid = [w["name"] for w in PANEL if WINDOW_VALID[w["name"]]]
print(f"Aggregate over valid-C windows: {_valid}\n")

_agg = []
for _key, _run in panel_runs.items():
    means = {k: float(np.mean([float(panel_evals[(_key, w)].challenge[k]) for w in _valid]))
             for k in KPI_COLS}
    _agg.append({"policy": _run["label"], **{k: round(v, 3) for k, v in means.items()}})
agg_df = pd.DataFrame(_agg).set_index("policy")
print("Panel aggregate KPIs:")
display(agg_df)

if "sac" in panel_runs:
    print("\nSLM performance as a fraction of the SAC expert")
    print("captured = (1 - SLM) / (1 - SAC), per window, mean over valid-C windows:\n")
    for _key, _run in panel_runs.items():
        if _key in ("noop", "random", "rbc", "sac"):
            continue
        line = f"  {_run['label']:30s}"
        for _metric, _tag in (("C  — cost", "cost C"), ("Phase I (C+G)/2", "Phase I")):
            fr = []
            for w in _valid:
                imp_sac = 1.0 - float(panel_evals[("sac", w)].challenge[_metric])
                imp_pol = 1.0 - float(panel_evals[(_key, w)].challenge[_metric])
                if imp_sac > 0:
                    fr.append(imp_pol / imp_sac)
            line += f"   {_tag}: {np.mean(fr)*100:6.1f}% of SAC" if fr else f"   {_tag}:   n/a"
        print(line)
    print("\nPhase I (= (C+G)/2) is the primary thesis metric; the >=70%-of-SAC gate applies to it.")

In [ ]:
# Challenge KPIs per window — bar chart
_kc = ["C  — cost", "G  — carbon", "R  — ramping", "1L — load factor"]
fig, axes = plt.subplots(1, len(PANEL), figsize=(5.2 * len(PANEL), 4.3), sharey=True)
for ax, w in zip(axes, PANEL):
    wn = w["name"]
    kpi_panel.xs(wn, level="window")[_kc].plot(
        kind="bar", ax=ax, width=0.8, legend=(wn == PANEL[-1]["name"]))
    ax.axhline(1.0, color="k", ls="--", lw=0.8)
    ax.set_title(f"{wn}  (month {window_month[wn]})"
                 f"{'' if WINDOW_VALID[wn] else '  — C degenerate'}", fontsize=9)
    ax.set_xlabel("")
    ax.tick_params(axis="x", labelrotation=80, labelsize=6)
axes[0].set_ylabel("ratio to no-control (lower better)")
plt.suptitle("Challenge KPIs per panel window", fontsize=10)
plt.tight_layout()
plt.show()

## § 11 — Per-Building Breakdown

Per-building action / SoC / energy stats across the whole panel — for SAC and the SLM.

In [ ]:
def _per_building(run: dict) -> pd.DataFrame:
    rows = []
    for b in range(N_BLDGS):
        a   = np.concatenate([wd["df"][f"a{b}"].values   for wd in run["windows"].values()])
        soc = np.concatenate([wd["df"][f"soc{b}"].values for wd in run["windows"].values()])
        net = np.concatenate([wd["df"][f"net{b}"].values for wd in run["windows"].values()])
        rew = np.concatenate([wd["df"][f"r{b}"].values   for wd in run["windows"].values()])
        rows.append({
            "run": run["label"], "building": f"B{b}",
            "total_reward": float(rew.sum()),
            "mean_soc_pct": float(soc.mean() * 100),
            "peak_net_kW":  float(net.max()),
            "mean_action":  float(a.mean()),
            "std_action":   float(a.std()),
        })
    return pd.DataFrame(rows)

_llm_keys = [k for k in panel_runs if k not in ("noop", "random", "rbc", "sac")]
_show = (["sac"] if "sac" in panel_runs else []) + _llm_keys
if _show:
    per_b = pd.concat([_per_building(panel_runs[k]) for k in _show], ignore_index=True)
    print(f"Per-building breakdown across the panel (buildings {BUILDINGS_RUN}):")
    display(per_b.set_index(["run", "building"]).round(4))
else:
    per_b = pd.DataFrame()
    print("No runs to break down yet.")

## § 12 — Diagnostics

SoC trajectories, district net load, sample raw responses, and decode-throughput stats — all panel-aware.

In [ ]:
# 12.1  SoC trajectories — rows = panel windows, cols = RBC / SAC / SLM
_cols = [(k, panel_runs[k]["label"]) for k in (["rbc", "sac"] + _llm_keys) if k in panel_runs]
if _cols:
    fig, axes = plt.subplots(len(PANEL), len(_cols),
                             figsize=(4.0 * len(_cols), 2.9 * len(PANEL)),
                             squeeze=False, sharey=True)
    _bc = ["#1f77b4", "#ff7f0e", "#2ca02c"]
    for ri, w in enumerate(PANEL):
        wn = w["name"]
        for ci, (k, lbl) in enumerate(_cols):
            ax  = axes[ri][ci]
            df_ = panel_runs[k]["windows"][wn]["df"]
            for b in range(N_BLDGS):
                ax.plot(df_["t"], df_[f"soc{b}"] * 100, lw=1.2, color=_bc[b], label=f"B{b}")
            peak = (df_["price"] >= PRICE_PEAK_THRESHOLD).values
            i = 0
            while i < len(peak):
                if peak[i]:
                    j = i
                    while j < len(peak) and peak[j]:
                        j += 1
                    ax.axvspan(i, j - 1, color="gold", alpha=0.13)
                    i = j
                else:
                    i += 1
            if ri == 0:
                ax.set_title(lbl, fontsize=9)
            if ci == 0:
                ax.set_ylabel(f"{wn} (m{window_month[wn]})\nSoC %", fontsize=8)
            ax.grid(alpha=0.3)
            ax.tick_params(labelsize=7)
    axes[0][0].legend(ncol=3, fontsize=6)
    plt.suptitle("SoC trajectories — rows = panel windows | gold = PEAK price", fontsize=10)
    plt.tight_layout()
    plt.show()
else:
    print("No runs yet.")

In [ ]:
# 12.2  District net load per panel window
_series = [("noop", "gray"), ("rbc", "steelblue"), ("sac", "forestgreen")]
_series += list(zip(_llm_keys, ["crimson", "darkorange", "purple"]))
fig, axes = plt.subplots(1, len(PANEL), figsize=(5.0 * len(PANEL), 3.6), sharey=True)
for ax, w in zip(axes, PANEL):
    wn = w["name"]
    for k, col in _series:
        if k not in panel_runs:
            continue
        df_ = panel_runs[k]["windows"][wn]["df"]
        net = df_[[f"net{i}" for i in range(N_BLDGS)]].sum(axis=1)
        ax.plot(df_["t"], net, lw=1.1, color=col, alpha=0.85, label=panel_runs[k]["label"])
    ax.axhline(0, color="k", lw=0.6)
    ax.set_title(f"{wn}  (month {window_month[wn]})", fontsize=9)
    ax.set_xlabel("hour t")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("district net load (kWh)")
axes[-1].legend(fontsize=6, ncol=2)
plt.suptitle("District net load per panel window  (positive = grid import)", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# 12.3  Sample raw SLM responses — one timestep per window
if _llm_keys:
    k = _llm_keys[0]
    for w in PANEL:
        log = panel_runs[k]["windows"][w["name"]]["raw_log"]
        if not log:
            continue
        e = log[len(log) // 2]
        print("=" * 70)
        print(f"{w['name']} (month {window_month[w['name']]})  t={e['t']}")
        print(f"State:\n{e['state_text']}")
        print(f"Response (fallback={e['fallback']}):\n{e['raw']}")
else:
    print("No SLM run yet.")

In [ ]:
# 12.4  Decode throughput + fallback rate across the panel
if _llm_keys:
    k = _llm_keys[0]
    raw = [r for wd in panel_runs[k]["windows"].values() for r in wd["raw_log"]]
    toks = [len(slm.tokenizer.encode(r["raw"], add_special_tokens=False))
            for r in raw if r["raw"]]
    fb = sum(1 for r in raw if r["fallback"])
    print(f"SLM            : {panel_runs[k]['label']}")
    print(f"Total calls    : {len(raw)}  ({len(PANEL)} windows x {PANEL_LEN})")
    print(f"Tokens/call    : mean={np.mean(toks):.0f}  min={min(toks)}  max={max(toks)}")
    print(f"Fallback rate  : {fb}/{len(raw)} ({100*fb/len(raw):.1f}%)  (target 0%)")
    _wtok = len(slm.tokenizer.encode(_resp, add_special_tokens=False))
    _tps  = _wtok / (_wt1 - _wt0) if (_wt1 - _wt0) > 0 else 0
    print(f"Throughput     : ~{_tps:.0f} tokens/s  (from the § 5 warmup)")
else:
    print("No SLM run yet.")

## § 13 — Panel Calibration

Does the 4-window panel proxy a full year? SAC is free to run, so we score it on the whole year and compare to its panel mean. A small gap means the panel is a trustworthy cheap proxy.

In [ ]:
if sac_agent is None:
    print("SAC not loaded — calibration skipped (set USE_SAC=True and provide the pickle).")
else:
    _df_yr, _env_yr = run_sac("sac_fullyear", sac_agent, start=0, length=8760,
                              env_factory=_panel_factory)
    _c_year  = float(evaluate(_env_yr, "SAC full-year").challenge["C  — cost"])
    _valid   = [w["name"] for w in PANEL if WINDOW_VALID[w["name"]]]
    _c_panel = float(np.mean([float(panel_evals[("sac", w)].challenge["C  — cost"])
                              for w in _valid]))
    print(f"\nSAC full-year C (buildings {BUILDINGS_RUN})       : {_c_year:.4f}")
    print(f"SAC panel-mean C (valid windows {_valid}) : {_c_panel:.4f}")
    print(f"Proxy error |panel - year|                  : {abs(_c_panel - _c_year):.4f}")
    print("\n-> a small error confirms the valid-C panel windows proxy the full year.")

## § 14 — Findings & Reading Guide

### Why a seasonal panel, not a single week

1. **The cost KPI degenerates in solar-rich windows.** `C` is a ratio — controlled cost / no-control cost. Where solar ≈ load the no-control district is roughly net-zero, the baseline cost collapses toward zero, and the ratio explodes or flips sign (a *negative* `C` is meaningless). `G`, `R`, `1L` never degenerate — emissions are floored at zero on export and ramping is absolute-valued. The panel flags degenerate windows (`C valid = NO`) and excludes them from the cost aggregate.
2. **A single window is a biased estimator.** Performance is strongly seasonal — winter offers a clear price-arbitrage opportunity, solar-saturated windows reward doing almost nothing. The 4-window panel averages over this; § 13 calibrates it against the full year.
3. **Compare like with like.** SLM, SAC, and the baselines are all scored on the *same* windows — the signal is the gap to SAC on identical data.

### How to read the results

- **§ 10 aggregate + capture-vs-SAC** is the headline. Phase I `(C+G)/2` is the primary thesis metric; the validation gate is **≥ 70% of SAC** on Phase I.
- Read **cost (`C`) and carbon (`G`) separately** — a model can tie SAC on cost while being far behind on carbon (the remote-API study in nb 02 found exactly that for DeepSeek: ≈100% of SAC on `C`, ≈30% on Phase I, because it was carbon-blind). The `G` column makes that visible.
- The **§ 12 SoC traces** reveal *how* the SLM uses the battery — healthy cycling vs "charge-and-park" (fill to ~85% then idle), which inflates carbon and the MERLIN SoC penalty.
- A high **fallback rate** (§ 12.4) means parse/timeout failures — the local SLM is not reliably emitting the action format, and the KPIs understate a wiring problem rather than a policy one.

### Implication for the thesis

This notebook establishes where the *local* zero-shot SLM sits relative to the SAC expert on the seasonal panel — the pre-fine-tuning baseline. nb 06 then measures whether the SAC→SLM distillation closes the gap.